# Question 1: Maximum Drawdown (Time Series / PnL)

You'll typically be given a sequence of portfolio values over time and asked to compute the maximum drawdown — the largest decline from a peak to a subsequent trough.

Approach: single pass, O(n) time, O(1) space.
- Track a running peak as you iterate.
- At each step, compute the drop from that peak to the current value and update your maximum drawdown if it's larger.
- If the current value exceeds the running peak, update the peak.

Key edge cases: empty array or single value → 0; monotonically increasing → 0; monotonically decreasing → drop from first to last value. If a percentage drawdown is asked for, compute (peak − current) / peak instead.



In [ ]:
def calculate_max_drawdown(values: list[float]) -> float:
    """
    Calculates the maximum drawdown of a sequence of portfolio values.
    """
    running_peak = values[0]
    max_drawdown = 0.0

    for current in values[1:]:
        # Update the running peak
        if current > running_peak:
            running_peak = current
            
        # Calculate the drawdown from the current peak
        else:
            
            current_drawdown = (running_peak - current) # add this for percentage: / running_peak

            # Update max drawdown if the current one is larger
            if current_drawdown > max_drawdown:
                max_drawdown = current_drawdown

    return max_drawdown

In [ ]:
def max_dd(values: list[float]) -> float:
    running_peak = values[0]
    max_dd = 0.0

    for i in values[1:]:
        if i > running_peak:

            running_peak = i
        else:
            dd = running_peak - i
            if dd > max_dd:
                max_dd = dd
    
    return max_dd

# Question 2: Trapping Rain Water (Array / Two-Pointer)

You'll be given an array of block heights and asked how many units of water are trapped between them after rain.

Core insight: water at any position is bounded by the shorter of the tallest bar to its left and the tallest bar to its right. Water depth = min(max_left, max_right) − current_height (floored at zero).

Approach progression to demonstrate:
1. Naïve – for each index, scan left and right for the tallest bar. Simple, but O(n²).
2. Prefix/suffix arrays – precompute left and right maxima in two passes, then a final pass for water. O(n) time, O(n) space.
3. Optimal (two-pointer) – use left and right pointers moving inward, tracking left_max and right_max. Always advance the pointer on the side with the lower current height, as that side's water contribution is deterministic. O(n) time, O(1) space.

Key edge case: array length < 3 → no water can be trapped.


## The Problem Statement

Given n non-negative integers representing an elevation map where the width of each bar is 1, compute how much water it can trap after raining.

Standard Examples Provided in the Prompt

### Example 1:
Input: height = [0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]
Output: 6
Explanation: In this elevation map, water is trapped at indices 2, 4, 5, 6, 9, and 10, totaling 6 units of water.

### Example 2:
Input: height = [4, 2, 0, 3, 2, 5]
Output: 9

### Typical Constraints:
You will also usually see execution constraints at the bottom of the prompt to hint at the required time complexity (which is why the $O(n^2)$ brute force fails hidden tests):
- n == height.length
- 1 <= n <= 2 * 10^4
- 0 <= height[i] <= 10^5

In [56]:
# n^2 solution
def trapped_water(heights):
    # water at any index = min(tallest block to left, tallest block to right) - index block height

    water = []

    for i,v in enumerate(heights):
        max_left = 0
        max_right = 0

        # print(i,v)
        
        if i != len(heights)-1:
            for j in heights[i+1:]:
                if j > max_right:
                    max_right = j
        
        if i != 0:
            for k in heights[i-1::-1]:
                if k > max_left:
                    max_left = k

        if v > min(max_left, max_right):
            water_at_i = 0
        else:
            water_at_i = min(max_left, max_right) - v

        water.append(water_at_i)

        print(i,v,"----",max_left,max_right,water_at_i)
    

    return water, sum(water)
        

In [57]:
height = [0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]

print(trapped_water(height))

0 0 ---- 0 3 0
1 1 ---- 0 3 0
2 0 ---- 1 3 1
3 2 ---- 1 3 0
4 1 ---- 2 3 1
5 0 ---- 2 3 2
6 1 ---- 2 3 1
7 3 ---- 2 2 0
8 2 ---- 3 2 0
9 1 ---- 3 2 1
10 2 ---- 3 1 0
11 1 ---- 3 0 0
([0, 0, 1, 0, 1, 2, 1, 0, 0, 1, 0, 0], 6)


In [55]:
height = [4, 2, 0, 3, 2, 5]
print(trapped_water(height))

0 4 ---- 0 5 0
1 2 ---- 4 5 2
2 0 ---- 4 5 4
3 3 ---- 4 5 1
4 2 ---- 4 5 2
5 5 ---- 4 0 0
([0, 2, 4, 1, 2, 0], 9)


### o(n) solution

In [58]:
def trapped_water_on(height):

    n = len(height)

    left_max = [0] * n
    right_max = [0] * n

    # left to right
    left_max[0] = height[0]
    for i in range(1, n):
        left_max[i] = max(left_max[i-1], height[i])

    # right to left
    right_max[n-1] = height[n-1]
    for i in range(n-2, -1, -1):
        right_max[i] = max(right_max[i+1], height[i])

    # calc water at each index
    total_water = 0
    for i in range(n):
        boundary = min(left_max[i], right_max[i])
        total_water += boundary - height[i]

    return total_water

In [60]:
height = [0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]

print(trapped_water_on(height))

height = [4, 2, 0, 3, 2, 5]
print(trapped_water_on(height))

6
9
